# Hands-on 30 (Bab 12): Pipeline Lengkap Sensor -> MQTT -> Python -> Prediksi -> Storage -> Dashboard

Buku Referensi: *IoT Terintegrasi dengan Machine Learning & Data Analytics* (Onno W. Purbo), Bab 12.

Pipeline ujung ke ujung Smart Room Monitoring: simulator mengirim telemetry ke MQTT,
subscriber Python melakukan preprocessing (lag + moving average dari buffer), model
klasifikasi memberi status (normal/warning/fault), hasil disimpan. Di buku storage
memakai InfluxDB + dashboard Grafana; di sini didemokan dengan storage CSV lokal
(InfluxDB opsional via `--influx`, butuh Docker).

In [1]:
import os
from pathlib import Path

while not (Path.cwd() / "scripts" / "mqtt").is_dir():
    os.chdir("..")
print("[INFO] Working directory:", Path.cwd())

# --- Utilitas notebook: menjalankan skrip buku APA ADANYA (tanpa flag) ---
# Skrip buku memakai while True / loop_forever, jadi notebook menghentikannya
# dari luar setelah beberapa detik dan menyimpan keluaran parsialnya.
import subprocess, sys, time
from pathlib import Path

LOG_DIR = Path("outputs/logs")
LOG_DIR.mkdir(parents=True, exist_ok=True)

def latar(script, nama_log, args=()):
    """Jalankan skrip buku di latar belakang (python -u), keluaran ke file log."""
    f = open(LOG_DIR / nama_log, "w", encoding="utf-8")
    p = subprocess.Popen([sys.executable, "-u", script, *args],
                         stdout=f, stderr=subprocess.STDOUT)
    return p, f

def henti(*pasangan):
    """Hentikan proses latar belakang dan tutup file lognya."""
    for p, f in pasangan:
        p.terminate()
        try:
            p.wait(timeout=5)
        except subprocess.TimeoutExpired:
            p.kill()
        f.close()

def jalankan_buku(script, detik, args=()):
    """Jalankan skrip buku selama `detik` detik lalu hentikan; kembalikan keluaran."""
    p = subprocess.Popen([sys.executable, "-u", script, *args],
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    try:
        out, _ = p.communicate(timeout=detik)
    except subprocess.TimeoutExpired:
        p.terminate()
        try:
            out, _ = p.communicate(timeout=5)
        except subprocess.TimeoutExpired:
            p.kill()
            out, _ = p.communicate()
    return out

def log_teks(nama_log):
    return (LOG_DIR / nama_log).read_text(encoding="utf-8")

# --- Utilitas InfluxDB (nilai bawaan skrip buku) ---
from influxdb_client import InfluxDBClient
INFLUX_URL, INFLUX_TOKEN = "http://localhost:8086", "GANTI_DENGAN_TOKEN_ANDA"
INFLUX_ORG, INFLUX_BUCKET = "iotlab", "sensor_data"
_ic = InfluxDBClient(url=INFLUX_URL, token=INFLUX_TOKEN, org=INFLUX_ORG)
if not _ic.ping():
    raise RuntimeError("InfluxDB belum berjalan di localhost:8086 (org iotlab, bucket sensor_data, "
                       "token GANTI_DENGAN_TOKEN_ANDA). Skrip buku di notebook ini menulis ke InfluxDB.")

def influx_df(measurement, rentang="-30d"):
    import pandas as pd
    q = (f'from(bucket:"{INFLUX_BUCKET}") |> range(start: {rentang}, stop: 1d) '
         f'|> filter(fn:(r) => r._measurement == "{measurement}") '
         f'|> pivot(rowKey:["_time"], columnKey:["_field"], valueColumn:"_value")')
    res = _ic.query_api().query_data_frame(q)
    df = pd.concat(res) if isinstance(res, list) else res
    if len(df):
        df = df.drop(columns=[c for c in df.columns if c in ("result", "table", "_start", "_stop", "_measurement")])
        df = df.sort_values("_time").reset_index(drop=True)
    return df

def influx_hapus(measurement):
    _ic.delete_api().delete("1970-01-01T00:00:00Z", "2100-01-01T00:00:00Z",
                            f'_measurement="{measurement}"', bucket=INFLUX_BUCKET, org=INFLUX_ORG)


[INFO] Working directory: /home/sandbox/iot-ml-book


## 1. Jalankan Pipeline End-to-End (broker + subscriber + simulator)

In [2]:
influx_hapus("room_pipeline")
broker = subprocess.Popen([sys.executable, "scripts/mqtt/run_broker.py"],
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(3)
try:
    sub = latar("scripts/pipeline_room_subscriber.py", "pipeline_sub_nb.log")
    time.sleep(2)
    pub = latar("scripts/publisher_room_simulator.py", "pipeline_pub_nb.log")
    time.sleep(24)   # sekitar 12 pesan (interval 2 detik)
    henti(pub, sub)
    print("--- Output publisher ---")
    print(log_teks("pipeline_pub_nb.log")[:600])
    print("--- Output subscriber pipeline ---")
    print(log_teks("pipeline_sub_nb.log"))
finally:
    broker.terminate()

--- Output publisher ---
Published: {"timestamp": "2026-10-22T13:27:17", "device_id": "room01", "temperature_c": 29.96, "humidity_pct": 63.09, "co2_ppm": 1350, "power_watt": 216.83}
Published: {"timestamp": "2026-10-22T13:27:19", "device_id": "room01", "temperature_c": 28.11, "humidity_pct": 42.17, "co2_ppm": 555, "power_watt": 226.41}
Published: {"timestamp": "2026-10-22T13:27:21", "device_id": "room01", "temperature_c": 27.91, "humidity_pct": 66.21, "co2_ppm": 909, "power_watt": 200.66}
Published: {"timestamp": "2026-10-22T13:27:23", "device_id": "room01", "temperature_c": 28.01, "humidity_pct": 66.7, "co2_ppm": 552
--- Output subscriber pipeline ---
Subscriber terhubung ke broker.
Subscribe ke: iotlab/room01/telemetry
[2026-10-22T13:27:17] T=29.96 H=63.09 CO2=1350 P=216.83 -> status=warning
[2026-10-22T13:27:19] T=28.11 H=42.17 CO2=555 P=226.41 -> status=normal
[2026-10-22T13:27:21] T=27.91 H=66.21 CO2=909 P=200.66 -> status=normal
[2026-10-22T13:27:23] T=28.01 H=66.7 CO2=552 P=243.

## 2. Verifikasi Hasil di Storage CSV

In [3]:
df = influx_df("room_pipeline")
print(df[["_time", "temperature_c", "humidity_pct", "co2_ppm", "power_watt", "pred_status"]].to_string(index=False))
print("\nDistribusi status:", df["pred_status"].value_counts().to_dict())

                    _time  temperature_c  humidity_pct  co2_ppm  power_watt pred_status
2026-10-22 13:27:17+00:00          29.96         63.09   1350.0      216.83     warning
2026-10-22 13:27:19+00:00          28.11         42.17    555.0      226.41      normal
2026-10-22 13:27:21+00:00          27.91         66.21    909.0      200.66      normal
2026-10-22 13:27:23+00:00          28.01         66.70    552.0      243.65      normal
2026-10-22 13:27:25+00:00          28.36         55.18    982.0      112.81      normal
2026-10-22 13:27:27+00:00          32.78         48.44    992.0      175.15     warning
2026-10-22 13:27:29+00:00          25.33         52.43   1130.0      158.61      normal
2026-10-22 13:27:31+00:00          31.48         63.57   1419.0      135.71     warning
2026-10-22 13:27:33+00:00          25.81         66.83    628.0      279.16      normal
2026-10-22 13:27:35+00:00          32.36         63.94   1091.0      178.28     warning
2026-10-22 13:27:37+00:00       

## Penutup

Seluruh komponen menyatu: MQTT menjadi pintu masuk pipeline analitik, subscriber
Python menjadi pusat logika (validasi, fitur lag/rolling dari buffer), model dipanggil
otomatis tanpa notebook, dan hasil inferensi tersimpan. Untuk deployment penuh sesuai
buku, jalankan InfluxDB 2 dan Grafana via Docker, aktifkan `--influx`, lalu susun panel
suhu/kelembaban/CO2/daya/status dengan ambang warna normal-hijau, warning-kuning,
fault-merah.

## Catatan dari buku: pengamatan, variasi, dan latihan

Bagian berikut disalin dari buku (Hands-on 32 dalam urutan buku) sebagai panduan pengamatan dan latihan mandiri setelah menjalankan notebook ini.

### 10. Apa yang dipelajari pembaca dari Hands-on 30?

Hands-on ini mengajarkan bahwa:

- MQTT bukan hanya untuk kirim data, tetapi menjadi pintu masuk ke pipeline
analitik,
- Python subscriber bisa menjadi pusat logika sistem,
- model bisa dipanggil otomatis tanpa notebook,
- hasil inferensi bisa disimpan dan langsung divisualisasikan,
- dan keseluruhan sistem dapat dijalankan secara lokal dengan stack open
source.